# Segment-level metrics with the calculated $\varepsilon$ &mdash; Condor edition

Mirror of §14e from
`Toy_Characterisation/Verify_new_results/segment_level_analysis.ipynb` (classical) and
`Quantum_segment_level_analysis.ipynb` (statevector 1BQF / OneBitHHL), but the angular
acceptance threshold is **recomputed from the formula** instead of the hand-tuned
value $\varepsilon = 2\,$mrad used in the source notebooks.

$$
\varepsilon = \sqrt{2\,(s\,\sigma_{\rm scatt})^2
 + 12\,\arctan^2\!\bigl(s\,\sigma_{\rm res}/\Delta z\bigr)
 + 2\,\theta_{\min}^2}
$$

with $\Delta z = 33$ mm, $s = 3$, $\theta_{\min} = 1.5\times 10^{-5}$ rad.

## Fixed parameters (first pass &mdash; match §14e exactly)

| Param | Value |
|---|---|
| $\sigma_{\rm res}$ | $5\,\mu$m ($5\times10^{-3}$ mm) |
| $\sigma_{\rm scatt}$ | $10^{-4}$ rad |
| drop rate | 1 % |
| $\phi_{\max}=\theta_{\max}$ | 0.2 rad |
| $\gamma,\delta$ | 3, 1 |
| Solver threshold $\tau$ | 0.35 (absolute) |

## Data flow

1. `gen_params_seg14e.py` writes per-memory-tier CSVs to `params/seg14e_calc_eps/`.
2. `submit_seg14e.sh` submits them through `_shared/submit_base.sub`, which runs
   `_shared/run_worker.py` once per `(T, rep)` row. The worker saves a pickle
   with `sol_C`, `sol_Q`, `truth`, the angle arrays, and the standard
   `*_metrics_default` dicts.
3. This notebook **reads only the pickles**: it recomputes the §14e count schema
   (`n_true_active`, `n_false_active`, `n_active`, `n_true_all`, `n_false_all`)
   from the saved vectors at the absolute threshold $\tau = 0.35$, aggregates per $T$,
   and plots.

In [ ]:
# §1  Setup
import os, sys, pickle, subprocess, shlex
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

REPO   = Path('/data/bfys/gscriven/Quantum_Track_Reconstruction')
WORK   = REPO / 'Toy_Characterisation' / 'Epsilon_study_2'
RESULTS_BASE = WORK / 'results' / 'seg14e_calc_eps'
RES_C  = RESULTS_BASE / 'classical'
RES_Q  = RESULTS_BASE / 'quantum'
FIGDIR = WORK / 'figures' / 'seg14e_calc_eps'
FIGDIR.mkdir(parents=True, exist_ok=True)

SHARED = REPO / 'Toy_Characterisation' / '_shared'
if str(SHARED) not in sys.path:
    sys.path.insert(0, str(SHARED))
from helpers import compute_epsilon  # noqa: E402

# Physics knobs (must match gen_params_seg14e.py)
SIGMA_RES   = 5e-3      # mm
SIGMA_SCATT = 1e-4      # rad
DROP_RATE   = 0.01
PHI_MAX     = 0.2
THRESHOLD   = 0.35      # absolute classical/quantum solver threshold (§14e)

EPSILON     = compute_epsilon(SIGMA_RES, SIGMA_SCATT)
EPSILON_OLD = 2e-3      # value baked into §14e of the source notebooks

print(f'sigma_res        = {SIGMA_RES:.3e} mm')
print(f'sigma_scatt      = {SIGMA_SCATT:.3e} rad')
print(f'epsilon (formula)= {EPSILON*1e3:.4f} mrad  ({EPSILON:.3e} rad)')
print(f'epsilon (old)    = {EPSILON_OLD*1e3:.4f} mrad   (for reference)')
print(f'ratio new/old    = {EPSILON/EPSILON_OLD:.3f}')
print()
print(f'classical pickles: {RES_C}')
print(f'quantum   pickles: {RES_Q}')

## §2  Submit to Condor

Run **once** at the start of the analysis. Re-run any time you want to add more
reps / track points (just edit `gen_params_seg14e.py`). The submit script is
idempotent: the worker skips any `(T, rep)` whose pickle already exists.

You can also run from a shell:

```bash
cd /data/bfys/gscriven/Quantum_Track_Reconstruction/Toy_Characterisation/Epsilon_study_2
./submit_seg14e.sh
```

In [ ]:
# §2  Submit (idempotent — comment out once submitted).
SUBMIT_SCRIPT = WORK / 'submit_seg14e.sh'
DO_SUBMIT = False  # flip to True the first time you run the notebook

if DO_SUBMIT:
    print(subprocess.check_output(['bash', str(SUBMIT_SCRIPT)], text=True))
else:
    print(f'Skipped. To submit run:  bash {SUBMIT_SCRIPT}')
    print('Track condor:           condor_q -nobatch')
    print('Resubmit without regen: bash {} --resubmit'.format(SUBMIT_SCRIPT))

In [ ]:
# §3  Sweep inventory — how much of the requested grid is on disk?
GEN_PARAMS_CSV_GLOB = sorted((WORK / 'params' / 'seg14e_calc_eps').glob('seg14e_*_mem*.csv'))

def _expected_rows(csv_paths):
    expected_C, expected_Q = {}, {}
    for p in csv_paths:
        for line in p.read_text().splitlines():
            parts = line.split(',')
            n_trk, rep, run_q = int(parts[0]), int(parts[1]), int(parts[12])
            target = expected_Q if run_q else expected_C
            target.setdefault(n_trk, set()).add(rep)
    return expected_C, expected_Q

def _have(pkl_dir):
    have = {}
    for p in sorted(Path(pkl_dir).glob('*.pkl')):
        try:
            r = pickle.load(open(p, 'rb'))
        except Exception:
            continue
        if r.get('status') != 'ok':
            continue
        have.setdefault(int(r['n_trk']), set()).add(int(r['rep']))
    return have

exp_C, exp_Q = _expected_rows(GEN_PARAMS_CSV_GLOB)
have_C = _have(RES_C)
have_Q = _have(RES_Q)

def _summarise(exp, have, label):
    rows = []
    for k in sorted(exp):
        rows.append(dict(solver=label, n_trk=k,
                         expected=len(exp[k]),
                         done=len(have.get(k, set())),
                         missing=len(exp[k] - have.get(k, set()))))
    return pd.DataFrame(rows)

inv = pd.concat([_summarise(exp_C, have_C, 'classical'),
                 _summarise(exp_Q, have_Q, 'quantum')], ignore_index=True)
print(inv.to_string(index=False))

In [ ]:
# §4  Per-event metric extraction.
#
# §14e uses an *absolute* threshold of 0.35; the worker stored its default
# metrics at the *relative* threshold tau*max(sol). We therefore recompute
# the count schema directly from the saved (sol_C, sol_Q, truth) vectors so
# this notebook's numbers match §14e regardless of the worker convention.

def _counts(sol, truth, threshold):
    sol = np.asarray(sol).ravel()
    truth = np.asarray(truth, dtype=bool).ravel()
    active = sol > threshold
    n_seg          = int(sol.size)
    n_true_all     = int(truth.sum())
    n_false_all    = n_seg - n_true_all
    n_true_active  = int((active & truth).sum())
    n_active       = int(active.sum())
    n_false_active = n_active - n_true_active
    return dict(
        n_seg=n_seg,
        n_true_all=n_true_all,
        n_false_all=n_false_all,
        n_true_active=n_true_active,
        n_false_active=n_false_active,
        n_active=n_active,
    )

def load_records(pkl_dir, threshold, solver):
    """Load all worker pickles in *pkl_dir* and recompute §14e counts at *threshold*.

    solver = 'C' -> use sol_C; solver = 'Q' -> use sol_Q.
    """
    rows = []
    for p in sorted(Path(pkl_dir).glob('*.pkl')):
        try:
            r = pickle.load(open(p, 'rb'))
        except Exception as exc:
            print(f'  WARN: failed to load {p.name}: {exc}')
            continue
        if r.get('status') != 'ok':
            continue
        sol  = r.get('sol_C' if solver == 'C' else 'sol_Q')
        truth = r.get('truth')
        if sol is None or truth is None:
            continue
        c = _counts(sol, truth, threshold)
        rows.append(dict(
            n_trk=int(r['n_trk']), rep=int(r['rep']),
            solver=solver,
            sigma_res=float(r.get('sigma_res', np.nan)),
            sigma_scatt=float(r.get('sigma_scatt', np.nan)),
            hit_ineff=float(r.get('hit_ineff', np.nan)),
            epsilon=float(r.get('epsilon', np.nan)),
            **c,
            n_qubits=r.get('n_qubits'),
            P_anc=r.get('P_anc'),
            cos_QC=r.get('cos_QC'),
            t_classical=r.get('t_classical'),
            t_q=r.get('t_q'),
        ))
    return pd.DataFrame(rows)

df_C = load_records(RES_C, THRESHOLD, 'C')
df_Q = load_records(RES_Q, THRESHOLD, 'Q')
print(f'classical events loaded: {len(df_C):4d}    unique T: {sorted(df_C["n_trk"].unique()) if len(df_C) else []}')
print(f'quantum   events loaded: {len(df_Q):4d}    unique T: {sorted(df_Q["n_trk"].unique()) if len(df_Q) else []}')

events_csv = WORK / 'results' / 'seg14e_calc_eps_events.csv'
pd.concat([df_C, df_Q], ignore_index=True).to_csv(events_csv, index=False)
print(f'wrote {events_csv}')

In [ ]:
# §5  Aggregation: mean + SEM per T  (matches §14e's _agg_solver layout).
def aggregate(df):
    if df.empty:
        return dict(tc=np.array([]), se_m=np.array([]), se_e=np.array([]),
                    fr_m=np.array([]), fr_e=np.array([]),
                    ntm=np.array([]), ntse=np.array([]),
                    nfm=np.array([]), nfse=np.array([]),
                    tam=np.array([]), tase=np.array([]),
                    fam=np.array([]), fase=np.array([]),
                    nrep=np.array([]))
    grid = []
    se_m, se_e, fr_m, fr_e = [], [], [], []
    ntm, ntse, nfm, nfse = [], [], [], []
    tam, tase, fam, fase = [], [], [], []
    nrep = []
    for k, sub in df.groupby('n_trk'):
        nr = len(sub)
        if nr == 0: continue
        # Efficiency denominator = n_true_all on this event
        # (no clean-vs-noisy distinction — see §1 notes).
        eff = (sub['n_true_active']  / sub['n_true_all'].clip(lower=1)).values * 100
        fr  = (sub['n_false_active'] / sub['n_active'].clip(lower=1)).values * 100
        sem = lambda a: a.std(ddof=1) / np.sqrt(nr) if nr > 1 else 0.0
        grid.append(k); nrep.append(nr)
        se_m.append(eff.mean()); se_e.append(sem(eff))
        fr_m.append(fr.mean());  fr_e.append(sem(fr))
        for col, m, e in [('n_true_all', ntm, ntse),
                          ('n_false_all', nfm, nfse),
                          ('n_true_active', tam, tase),
                          ('n_false_active', fam, fase)]:
            v = sub[col].values.astype(float)
            m.append(v.mean()); e.append(sem(v))
    return dict(
        tc=np.array(grid, dtype=float),
        nrep=np.array(nrep),
        se_m=np.array(se_m), se_e=np.array(se_e),
        fr_m=np.array(fr_m), fr_e=np.array(fr_e),
        ntm=np.array(ntm), ntse=np.array(ntse),
        nfm=np.array(nfm), nfse=np.array(nfse),
        tam=np.array(tam), tase=np.array(tase),
        fam=np.array(fam), fase=np.array(fase),
    )

agg_C = aggregate(df_C)
agg_Q = aggregate(df_Q)
print('classical T : ', agg_C['tc'].astype(int).tolist())
print('         reps:', agg_C['nrep'].tolist())
print('quantum   T : ', agg_Q['tc'].astype(int).tolist())
print('         reps:', agg_Q['nrep'].tolist())

## §6  §14e mirror &mdash; classical only, calculated $\varepsilon$

In [ ]:
# §6  Classical-only 2x2 plot (matches §14e layout / palette exactly).
plt.rcParams.update({
    'font.size': 12, 'axes.labelsize': 14, 'axes.titlesize': 14,
    'xtick.labelsize': 12, 'ytick.labelsize': 12, 'legend.fontsize': 10,
    'axes.linewidth': 1.2, 'lines.linewidth': 2, 'lines.markersize': 7,
    'axes.grid': True, 'grid.alpha': 0.3,
})

def _logclip_yerr(y, e, floor):
    y = np.asarray(y, dtype=float); e = np.asarray(e, dtype=float)
    lo = np.minimum(e, np.maximum(y - floor, 0.0))
    return np.vstack([lo, e])

c_drop  = '#e31a1c'
c_fr    = '#c51b7d'
c_true  = '#2166ac'
c_false = '#d6604d'

d1 = agg_C; tc = d1['tc']
fig, axes = plt.subplots(2, 2, figsize=(12, 10))

ax = axes[0, 0]
ax.errorbar(tc, d1['se_m'], yerr=d1['se_e'], fmt='o-', color=c_drop,
            capsize=5, capthick=1.5, markeredgecolor='black',
            markeredgewidth=0.8, zorder=3)
ax.axhline(100, color='gray', linestyle='--', linewidth=1.2, alpha=0.7, zorder=2)
ax.set_ylabel('Segment Efficiency (%)', fontsize=11)
ax.set_title(r'i) Segment Efficiency ($N_{\rm true\,act} / N_{\rm true\,all}$)', fontweight='bold')
ax.yaxis.set_major_formatter(mticker.FormatStrFormatter('%g%%'))
ax.grid(True, alpha=0.3); ax.tick_params(which='both', direction='in', top=True, right=True)

ax = axes[0, 1]
ax.errorbar(tc, d1['fr_m'], yerr=d1['fr_e'], fmt='s-', color=c_fr,
            capsize=5, capthick=1.5, markeredgecolor='black',
            markeredgewidth=0.8, zorder=3)
ax.set_ylabel('Segment False Rate (%)', fontsize=11)
ax.set_title(r'ii) Segment False Rate ($N_{\rm false\,act} / N_{\rm accepted}$)', fontweight='bold')
ax.yaxis.set_major_formatter(mticker.FormatStrFormatter('%g%%'))
ax.grid(True, alpha=0.3); ax.tick_params(which='both', direction='in', top=True, right=True)

_F3 = 0.5
ax = axes[1, 0]
ax.errorbar(tc, d1['ntm'], yerr=_logclip_yerr(d1['ntm'], d1['ntse'], _F3),
            fmt='o-', color=c_true, capsize=4, capthick=1.2,
            markeredgecolor='black', markeredgewidth=0.8,
            label='True segments', zorder=3)
ax.errorbar(tc, d1['nfm'], yerr=_logclip_yerr(d1['nfm'], d1['nfse'], _F3),
            fmt='s-', color=c_false, capsize=4, capthick=1.2,
            markeredgecolor='black', markeredgewidth=0.8,
            label='False segments', zorder=3)
ax.set_yscale('log'); ax.set_ylabel('Number of Segment Pairs', fontsize=11)
ax.set_title('iii) Segment Pair Counts', fontweight='bold')
ax.legend(loc='upper left', framealpha=0.95, fontsize=10)
ax.grid(True, alpha=0.3, which='both'); ax.minorticks_on()
ax.tick_params(which='both', direction='in', top=True, right=True)

_F4 = 0.5
ax = axes[1, 1]
fa_p = np.maximum(d1['fam'], _F4)
fa_e = np.where(d1['fam'] >= _F4, d1['fase'], 0.0)
ax.errorbar(tc, d1['tam'], yerr=_logclip_yerr(d1['tam'], d1['tase'], _F4),
            fmt='o-', color=c_true, capsize=4, capthick=1.2,
            markeredgecolor='black', markeredgewidth=0.8,
            label='True active', zorder=3)
ax.errorbar(tc, fa_p, yerr=_logclip_yerr(fa_p, fa_e, _F4),
            fmt='s-', color=c_false, capsize=4, capthick=1.2,
            markeredgecolor='black', markeredgewidth=0.8,
            label='False active', zorder=3)
ax.set_yscale('log'); ax.set_ylim(_F4 * 0.7, None)
ax.set_ylabel('Number of Active Segments', fontsize=11)
ax.set_title('iv) Active Segment Pairs', fontweight='bold')
ax.legend(loc='lower right', framealpha=0.95, fontsize=10)
ax.grid(True, alpha=0.3, which='both'); ax.minorticks_on()
ax.tick_params(which='both', direction='in', top=True, right=True)

for ax in axes.flat:
    ax.set_xlabel('Number of Tracks', fontsize=11)
for ax in axes[1, :]:
    ax.set_xscale('log')

fig.suptitle(
    rf'Classical $\S14e$ mirror at calculated $\varepsilon = {EPSILON*1e3:.4f}$ mrad'
    rf'  (drop = {DROP_RATE*100:.0f}%, $\sigma_r$ = {SIGMA_RES*1e3:.1f} $\mu$m, '
    rf'$\sigma_s$ = {SIGMA_SCATT*1e3:.2f} mrad)',
    fontsize=13)
fig.tight_layout(rect=[0, 0, 1, 0.97])
stem = 'fig14e_classical_calc_eps_drop1pct'
fig.savefig(FIGDIR / f'{stem}.pdf', dpi=600, bbox_inches='tight', facecolor='white')
fig.savefig(FIGDIR / f'{stem}.png', dpi=300, bbox_inches='tight', facecolor='white')
plt.show()
print(f'saved -> {FIGDIR/stem}.pdf')

## §7  §14e mirror &mdash; classical + quantum overlay

Same layout as §6 but with the statevector 1BQF (OneBitHHL) curves overlaid.
Quantum is capped at $T = 200$ (statevector RAM); classical extends to $T = 1000$.

In [ ]:
# §7  Classical + quantum overlay (§14e layout)
c14_C  = '#2166ac'   # classical (blue diamonds)
c14_Q  = '#d62728'   # quantum   (red circles)
c14_t  = '#1b7837'
c14_f  = '#c51b7d'

fig, axes = plt.subplots(2, 2, figsize=(13, 11))

def _eb(ax, d, ym, ye, fmt, color, label, mfc=None, capsize=5):
    if d['tc'].size == 0: return
    kw = dict(fmt=fmt, color=color, capsize=capsize, capthick=1.5,
              markeredgecolor='black', markeredgewidth=0.8, zorder=3, label=label)
    if mfc is not None: kw['mfc'] = mfc
    ax.errorbar(d['tc'], d[ym], yerr=d[ye], **kw)

# i) efficiency
ax = axes[0, 0]
_eb(ax, agg_C, 'se_m', 'se_e', 'D--', c14_C, 'Classical solver', mfc='white')
_eb(ax, agg_Q, 'se_m', 'se_e', 'o-',  c14_Q, 'OneBitHHL (statevector)')
ax.axhline(100, color='gray', linestyle='--', linewidth=1.2, alpha=0.7, zorder=2)
ax.set_ylabel('Segment Efficiency (%)')
ax.set_title(r'i) Segment Efficiency ($N_{\rm true\,act}/N_{\rm true\,all}$)', fontweight='bold')
ax.yaxis.set_major_formatter(mticker.FormatStrFormatter('%g%%'))
ax.legend(loc='lower left', framealpha=0.95)
ax.grid(True, alpha=0.3); ax.tick_params(which='both', direction='in', top=True, right=True)

# ii) false rate
ax = axes[0, 1]
_eb(ax, agg_C, 'fr_m', 'fr_e', 'D--', c14_C, 'Classical solver', mfc='white')
_eb(ax, agg_Q, 'fr_m', 'fr_e', 'o-',  c14_Q, 'OneBitHHL (statevector)')
ax.set_ylabel('Segment False Rate (%)')
ax.set_title(r'ii) Segment False Rate ($N_{\rm false\,act}/N_{\rm accepted}$)', fontweight='bold')
ax.yaxis.set_major_formatter(mticker.FormatStrFormatter('%g%%'))
ax.legend(loc='upper left', framealpha=0.95)
ax.grid(True, alpha=0.3); ax.tick_params(which='both', direction='in', top=True, right=True)

# iii) pair counts (solver-independent; use the wider classical grid)
_F3 = 0.5
ax = axes[1, 0]
d = agg_C
if d['tc'].size:
    ax.errorbar(d['tc'], d['ntm'], yerr=_logclip_yerr(d['ntm'], d['ntse'], _F3),
                fmt='o-', color=c14_t, capsize=4, capthick=1.2,
                markeredgecolor='black', markeredgewidth=0.8,
                label='True segments', zorder=3)
    ax.errorbar(d['tc'], d['nfm'], yerr=_logclip_yerr(d['nfm'], d['nfse'], _F3),
                fmt='s-', color=c14_f, capsize=4, capthick=1.2,
                markeredgecolor='black', markeredgewidth=0.8,
                label='False segments', zorder=3)
ax.set_yscale('log'); ax.set_ylabel('Number of Segment Pairs')
ax.set_title('iii) Segment Pair Counts', fontweight='bold')
ax.legend(loc='upper left', framealpha=0.95)
ax.grid(True, alpha=0.3, which='both'); ax.minorticks_on()
ax.tick_params(which='both', direction='in', top=True, right=True)

# iv) active pair counts (classical vs quantum)
_F4 = 0.5
ax = axes[1, 1]
def _eb_act(d, ym, ye, fmt, color, label, mfc=None):
    if d['tc'].size == 0: return
    y  = d[ym]; e = d[ye]
    yp = np.maximum(y, _F4)
    ep = np.where(y >= _F4, e, 0.0)
    kw = dict(fmt=fmt, color=color, capsize=4, capthick=1.2,
              markeredgecolor='black', markeredgewidth=0.8, zorder=3, label=label)
    if mfc is not None: kw['mfc'] = mfc
    ax.errorbar(d['tc'], yp, yerr=_logclip_yerr(yp, ep, _F4), **kw)

_eb_act(agg_C, 'tam', 'tase', 'D--', c14_C, 'True active  - classical', mfc='white')
_eb_act(agg_Q, 'tam', 'tase', 'o-',  c14_Q, 'True active  - quantum')
_eb_act(agg_C, 'fam', 'fase', 'D:',  c14_f, 'False active - classical', mfc='white')
_eb_act(agg_Q, 'fam', 'fase', 'o:',  '#7b3294', 'False active - quantum')
ax.set_yscale('log'); ax.set_ylim(_F4 * 0.7, None)
ax.set_ylabel('Number of Active Segments')
ax.set_title('iv) Active Segment Pairs', fontweight='bold')
ax.legend(loc='upper left', framealpha=0.95, fontsize=8, ncol=2)
ax.grid(True, alpha=0.3, which='both'); ax.minorticks_on()
ax.tick_params(which='both', direction='in', top=True, right=True)

for ax in axes.flat:
    ax.set_xlabel('Number of Tracks')
for ax in axes[1, :]:
    ax.set_xscale('log')

fig.suptitle(
    rf'$\S14e$ mirror at calculated $\varepsilon = {EPSILON*1e3:.4f}$ mrad'
    rf'   (drop = {DROP_RATE*100:.0f}%, $\sigma_r$ = {SIGMA_RES*1e3:.1f} $\mu$m, '
    rf'$\sigma_s$ = {SIGMA_SCATT*1e3:.2f} mrad)',
    fontsize=13)
fig.tight_layout(rect=[0, 0, 1, 0.97])
stem = 'fig14e_classical_vs_quantum_calc_eps_drop1pct'
fig.savefig(FIGDIR / f'{stem}.pdf', dpi=600, bbox_inches='tight', facecolor='white')
fig.savefig(FIGDIR / f'{stem}.png', dpi=300, bbox_inches='tight', facecolor='white')
plt.show()
print(f'saved -> {FIGDIR/stem}.pdf')

In [ ]:
# §8  Per-T summary table
def _tbl(df, label):
    rows = []
    for k, sub in df.groupby('n_trk'):
        nr = len(sub)
        if nr == 0: continue
        eff = (sub['n_true_active']  / sub['n_true_all'].clip(lower=1)).values * 100
        fr  = (sub['n_false_active'] / sub['n_active'].clip(lower=1)).values * 100
        rows.append(dict(
            solver=label, n_trk=int(k), n_reps=nr,
            eff_mean=eff.mean(),
            eff_sem=eff.std(ddof=1) / np.sqrt(nr) if nr > 1 else 0.0,
            fr_mean=fr.mean(),
            fr_sem=fr.std(ddof=1) / np.sqrt(nr) if nr > 1 else 0.0,
            n_seg_mean=sub['n_seg'].mean(),
        ))
    return pd.DataFrame(rows)

tbl = pd.concat([_tbl(df_C, 'classical'),
                 _tbl(df_Q, 'quantum_sv')], ignore_index=True)
tbl.to_csv(WORK / 'results' / 'seg14e_calc_eps_summary.csv', index=False)
print(tbl.to_string(index=False, float_format=lambda v: f'{v:8.3f}'))
print(f"\nsaved -> {WORK / 'results' / 'seg14e_calc_eps_summary.csv'}")